# Slice Stack Alignment, Reslice, and Analysis

This notebook demonstrates a compact image-processing workflow built on
[SimpleITK](https://simpleitk.org/):

1. **Load** a stack of 2D slices (e.g. serial sections or a scan series) from a folder.
2. **Align** the slices to one another with rigid-body (2D Euler) registration so that
   features stay in correspondence from slice to slice.
3. **Reconstruct** the aligned slices into a 3D volume and visualize the stack, which
   is the basis for orthogonal reslicing and downstream analysis.

The physical dimensions used below (field of view, slice thickness) are placeholders
suitable for the example data set. Replace them with the true acquisition geometry for
your own images.

## Requirements

```bash
pip install SimpleITK numpy matplotlib
```

## Setup

All imports are consolidated here so the rest of the notebook can be run cell by cell.

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

# `Axes3D` registers the '3d' projection used later for the stack visualization.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

## 1. Load the image stack

`load_sitk_images_from_folder` reads every supported image in a folder, sorts them by
filename (so use zero-padded, sequential names such as `slice_001.png`, `slice_002.png`,
...), and returns them as single-channel 32-bit float SimpleITK images. Registration
requires scalar, floating-point images, so color images are collapsed to intensity and
all images are cast to `Float32`.

In [ ]:
def load_sitk_images_from_folder(folder, extensions=(".png", ".tif", ".tiff", ".jpg", ".jpeg")):
    """Load all images in `folder` as SimpleITK float images, sorted by filename.

    Parameters
    ----------
    folder : str
        Directory containing the slice images.
    extensions : tuple of str
        Filename extensions to include (case-insensitive).

    Returns
    -------
    list of SimpleITK.Image
        One 2D, single-channel Float32 image per file, in filename order.
    """
    filenames = sorted(
        name for name in os.listdir(folder)
        if name.lower().endswith(extensions)
    )
    if not filenames:
        raise FileNotFoundError(f"No image files found in {folder!r}.")

    images = []
    for name in filenames:
        image = sitk.ReadImage(os.path.join(folder, name))

        # Collapse multi-channel (e.g. RGB/RGBA) images to a single intensity channel.
        if image.GetNumberOfComponentsPerPixel() > 1:
            image = sitk.VectorMagnitude(image)

        # Registration operates on floating-point images.
        image = sitk.Cast(image, sitk.sitkFloat32)
        images.append(image)

    return images

## 2. Rigid-body alignment

Each slice after the first is registered to the **first slice** (the fixed reference)
using a 2D Euler transform (rotation + translation). The registration uses a
multi-resolution strategy for robustness and mean-squares as the similarity metric,
which is appropriate when slices share the same intensity characteristics.

The first slice is kept unchanged; every other slice is resampled into the reference
frame, so all returned slices share a common coordinate system.

In [ ]:
def align_images_sitk(images, default_pixel_value=100):
    """Align a series of images to the first image using 2D rigid-body registration.

    Parameters
    ----------
    images : list of SimpleITK.Image
        Slices to align. The first image is used as the fixed reference.
    default_pixel_value : float
        Value used to fill regions that fall outside the moving image after resampling.

    Returns
    -------
    list of SimpleITK.Image
        The aligned slices, in the same order, resampled onto the reference grid.
    """
    fixed_image = images[0]
    aligned_images = [fixed_image]

    for moving_image in images[1:]:
        # Initialize the transform by aligning the geometric centers of the two images.
        initial_transform = sitk.CenteredTransformInitializer(
            fixed_image,
            moving_image,
            sitk.Euler2DTransform(),
            sitk.CenteredTransformInitializerFilter.GEOMETRY,
        )

        registration_method = sitk.ImageRegistrationMethod()

        # Similarity metric: mean squares, evaluated on a random 10% sample of pixels.
        registration_method.SetMetricAsMeanSquares()
        registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
        registration_method.SetMetricSamplingPercentage(0.1)

        registration_method.SetInterpolator(sitk.sitkLinear)

        # Gradient-descent optimizer with scales estimated from physical shifts.
        registration_method.SetOptimizerAsGradientDescent(
            learningRate=1.0,
            numberOfIterations=100,
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10,
        )
        registration_method.SetOptimizerScalesFromPhysicalShift()

        # Multi-resolution (coarse-to-fine) framework for robustness and speed.
        registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
        registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
        registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

        # Register into a copy of the transform so the initial transform is preserved.
        registration_method.SetInitialTransform(initial_transform, inPlace=False)

        final_transform = registration_method.Execute(fixed_image, moving_image)

        # Resample the moving image onto the reference (fixed) image grid.
        resampler = sitk.ResampleImageFilter()
        resampler.SetReferenceImage(fixed_image)
        resampler.SetTransform(final_transform)
        resampler.SetInterpolator(sitk.sitkLinear)
        resampler.SetDefaultPixelValue(default_pixel_value)

        aligned_image = resampler.Execute(moving_image)
        aligned_images.append(aligned_image)

    return aligned_images

### Run the alignment

Point `folder` at the directory holding your slice images, then load, align, and preview
the result. Update the path below to match your own data.

In [ ]:
# Folder containing the ordered slice images. Update this path to your own data.
folder = "/path/to/your/slice_images"

# Load and align the stack.
sitk_images = load_sitk_images_from_folder(folder)
aligned_sitk_images = align_images_sitk(sitk_images)

# Convert to NumPy arrays for visualization with matplotlib.
aligned_numpy_images = [sitk.GetArrayFromImage(img) for img in aligned_sitk_images]

# Preview the first few aligned slices as a sanity check.
num_preview = min(9, len(aligned_numpy_images))
plt.figure(figsize=(15, 10))
for i, img in enumerate(aligned_numpy_images[:num_preview]):
    plt.subplot(1, num_preview, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Aligned Image {i + 1}")
    plt.axis("off")
plt.show()

## 3. 3D visualization of the aligned stack

With the slices aligned, they can be stacked into a volume. Each slice is drawn as a
textured plane at its physical depth, giving a 3D view of the reconstructed stack. This
volume is the starting point for orthogonal reslicing (viewing the data along the XZ or
YZ planes) and further quantitative analysis.

`fov_x_mm` and `fov_y_mm` are the in-plane physical dimensions (field of view) of each
slice, and `slice_thickness_mm` is the spacing between consecutive slices. Set these to
match your acquisition.

In [ ]:
def create_3d_visualization(aligned_images, fov_x_mm=300, fov_y_mm=160, slice_thickness_mm=2):
    """Render an aligned image stack as a set of textured planes in 3D.

    Parameters
    ----------
    aligned_images : list of numpy.ndarray
        Aligned slices as 2D arrays, ordered from first to last.
    fov_x_mm, fov_y_mm : float
        In-plane physical size (field of view) of each slice, in millimeters.
    slice_thickness_mm : float
        Spacing between consecutive slices along the stacking (Z) axis, in millimeters.
    """
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection="3d")

    # Physical Z position of each slice along the stacking axis.
    num_slices = len(aligned_images)
    slice_positions = np.arange(0, num_slices * slice_thickness_mm, slice_thickness_mm)

    for i, slice_img in enumerate(aligned_images):
        # Build the in-plane (X, Y) grid in physical (mm) coordinates.
        X, Y = np.meshgrid(
            np.linspace(0, fov_x_mm, slice_img.shape[1]),
            np.linspace(0, fov_y_mm, slice_img.shape[0]),
        )
        Z = np.ones_like(X) * slice_positions[i]

        # Normalize intensities to [0, 1] for the grayscale colormap.
        max_value = np.max(slice_img)
        normalized_img = slice_img / max_value if max_value > 0 else slice_img

        # Draw the slice as a surface at depth Z, textured with the image intensities.
        ax.plot_surface(
            X, Z, Y,
            rstride=1, cstride=1,
            facecolors=plt.cm.gray(normalized_img),
            shade=False,
        )

    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Z (mm)")
    ax.set_zlabel("Y (mm)")
    ax.set_title("3D Visualization of Aligned Image Stack")

    plt.show()


# Visualize the aligned 3D stack.
create_3d_visualization(aligned_numpy_images)